In [ ]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEFAULT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"


class ResearchRAG:
    def __init__(self, retriever, model_name=DEFAULT_MODEL):
        self.retriever = retriever
        self.model_name = model_name

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        if torch.cuda.is_available():
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto"
            )
        else:
            self.model = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float32
            )

        self.model.eval()

    def build_prompt(self, question, passages):
        evidence_blocks = []

        for i, passage in enumerate(passages, start=1):
            evidence_blocks.append(
                f"[E{i}] Document: {passage['document']}\n"
                f"Page: {passage['page']}\n"
                f"Text: {passage['text']}"
            )

        evidence = "\n\n".join(evidence_blocks)

        return f"""
You are ResearchMate, an assistant that answers questions about research papers.

Rules:
- Answer using only the evidence supplied below.
- If the evidence does not answer the question, say that the available
  passages do not contain enough information.
- Do not invent facts, document names, or page numbers.
- Refer to supporting evidence using labels such as [E1] or [E2].
- Keep the answer clear and concise.

Evidence:
{evidence}

Question:
{question}

Answer:
""".strip()

    def answer(self, question, top_k=4, use_reranker=False):
        candidates = self.retriever.search_hybrid(
            question,
            top_k=10,
            candidate_k=15
        )

        if use_reranker and candidates:
            passages = self.retriever.rerank(
                question, candidates, top_k=top_k
            )
        else:
            passages = candidates[:top_k]

        if not passages:
            return {
                "answer": "No relevant passages were found.",
                "sources": [],
                "prompt": ""
            }

        prompt = self.build_prompt(question, passages)

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a careful research assistant. "
                    "Do not invent evidence or citations."
                )
            },
            {"role": "user", "content": prompt}
        ]

        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer(
            formatted_prompt,
            return_tensors="pt",
            truncation=True,
            max_length=6000
        )

        input_device = next(self.model.parameters()).device
        inputs = {key: value.to(input_device)
                  for key, value in inputs.items()}

        with torch.inference_mode():
            output = self.model.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=self.tokenizer.eos_token_id
            )

        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        answer_text = self.tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

        return {
            "answer": answer_text,
            "sources": passages,
            "prompt": prompt
        }
